# 노드 B — tonight

**6 GPU = 3노드 x 2 GPU. 이 노트북은 노드 B 전용.**

세 노드 전부 `exp5_tonight.py` 하나만 부른다. 잡 정의 · 우선순위 · 스킵 · 집계가 전부 거기 있다.
**세 노트북은 내용이 동일하고 노드 문자만 다르다** — 학습으로 배정되든 eval로 배정되든 이 하나로 된다.

**예산**: eval 1셀(LIBERO-10 x 50ep/task = 500ep) ≈ 2 GPU-h · 학습 1잡(150k) ≈ 8 GPU-h (8/24 밤 실측)

---

## 승리 조건 — 토너먼트 (8/25 재설계)

"ACT를 넘는다" = **각 방법에 같은 탐색 예산(K × stride × TE on/off)을 주고 방법별 best끼리 비교.**
우리 best 후보만 재고 ACT는 한 세팅만 재면 체리피킹이라 리뷰에서 깨진다.

- 우리 최고 후보: BiMamba+TE **K=10 → 75.4, K=15 → 75.6, K=20 → 71.4** (은지님 K sweep)
- ACT 알려진 최고: **67.6** (K=50, s=10, TE off)
- **판정 셀 = act+TE @ K=10/15/20 — 전부 비어 있다.**
  K=20은 act ckpt가 있어 **eval 하나로 오늘 판정**. K=10/15는 학습 필요(critical).

## 흐름

1. 부팅 → 2. 인벤토리 → 3. **역할 배정** → 4. 계획 → 5. preflight(한 번만)
→ **6. eval** 또는 **7. 학습** (3번이 정해준 쪽만) → 8. 이어서 → 집계

6)·7) 셀은 배정된 역할이 아니면 스스로 건너뛴다. 잘못 눌러도 사고 안 난다.

## 우선순위 (`exp5_tonight._all_jobs` / `TRAIN_PRIORITY`)

**eval**
1. TE 토너먼트 축 — K=20, 50 (즉시), K=10, 15 (critical 학습 후 자동 편입), K=100, 150
2. K=50 s=10/25 레짐 — ACT 최고점 67.6과 정면 승부 + carry 표 모순(69.5 vs 64.6) 해소
3. K=100 레짐 (긴 stride 우선) — 폴백 스토리(긴 chunk 크로스오버 + 효율)

**학습 (critical — eval 큐 깊이와 무관하게 노드 선점, `suggest()`가 처리)**
- `act_k10` / `act_k15` — 우리 75.4/75.6의 직접 상대. 이 열이 비면 토너먼트 주장 불가
- `bimamba_pure_k10` / `_k15` — 75.4를 UBAI·순수판으로 재측정 (은지님 값은 오염판+다른 환경)

**리스크**: 짧은 K+TE는 LIBERO 표준 세팅 = ACT 홈그라운드. act+TE가 다 이기면
폴백 = 긴 chunk 크로스오버(+효율) + real-world(SO-101 sorting·OOD). 내일이면 판정된다.


## 1) 부팅

In [ ]:
import sys
from pathlib import Path

_h = Path.cwd()
_r = next(c for c in (_h, *_h.parents) if (c / 'notebooks' / 'libero' / 'exp5_tonight.py').exists())
for _p in (_r / 'notebooks', _r / 'notebooks' / 'libero'):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

import importlib
import exp5_tonight as X
X = importlib.reload(X)
cf, v23 = X.setup()          # common_final reload + 태그 등록 (순서 중요)

NODE = 'B'
# 이 노드에서 쓸 GPU. 한 노드에서 창을 2개 띄울 때만 [2, 3] 처럼 직접 지정.
GPUS = v23.available_gpus([0, 1])
print('NODE', NODE, '| GPUS', GPUS)


## 2) 인벤토리 — 뭐가 학습돼 있고 뭐가 없나

서버 파일시스템을 실제로 스캔한다. `MISS` = 학습 필요, `PART` = 중단됨(resume 대상).
`MISS`/`PART`인 태그를 참조하는 eval 셀은 자동으로 큐에서 빠진다.

In [ ]:
rows = X.inventory()


## 3) 역할 배정 ← **여기가 오늘 뭘 할지 정한다**

규칙 (8/25):
1. **critical 학습(act/pure @ K=10/15)이 노드를 먼저 선점한다** — 토너먼트 판정 셀을
   여는 학습은 eval 큐 소화보다 가치가 높다. 끝나면 선점이 자동으로 풀린다.
2. 남는 노드는 ready eval 큐를 포화시킨다.
3. 그래도 남으면 일반 학습.

출력이 "기본값과 다르다"면 알려주는 `X.ROLE_OVERRIDE = ...` 한 줄을 **세 노트북 모두**에 넣어야 잡이 안 겹친다.

In [ ]:
roles = X.suggest()

# 위 출력이 "기본값과 다르다" 라고 하면, 알려주는 한 줄을 **세 노트북 모두**에 붙여넣고
# 이 셀부터 다시 실행할 것. (세 노드가 같은 역할표를 봐야 잡이 안 겹친다)
# X.ROLE_OVERRIDE = {'C': 'train'}


## 4) 계획 — 이 노드 몫

실행 전에 목록을 눈으로 확인할 것.

In [ ]:
plan = X.plan(NODE, GPUS)


## 5) preflight (약 12분) — **셋 중 한 노드에서 한 번만**

`--policy.n_action_steps` / `--policy.temporal_ensemble_coeff` override가 `lerobot_eval`에서 실제로 먹는지 확인한다. 여기서 죽으면 override 경로가 막힌 것이고, 그러면 조합별 학습이 필요해져 **계획을 전면 수정**해야 한다.

다른 노드에서 이미 통과했으면 건너뛸 것.

In [ ]:
X.preflight(gpu=GPUS[0], n_ep=5)


## 6) eval — 3)에서 `eval`로 배정됐을 때

먼저 dry-run으로 커맨드를 보고 실행. `GPUS` 수만큼 청크로 돌고 청크마다 블로킹한다.
로그는 `outputs/final/_logs/exp5__*.log`. **아침에 다시 실행하면 완료분은 skip되고 이어서 돈다.**

In [ ]:
X.run_evals(plan['eval'][:2], plan['gpus'], dry=True)


In [ ]:
if X.role_of(NODE) != 'eval':
    print('노드 ' + NODE + ' 는 train 으로 배정됐다 -> 7)번 학습 셀을 쓸 것. 여기는 건너뛴다.')
else:
    X.run_evals(plan['eval'], plan['gpus'])


## 7) 학습 — 3)에서 `train`으로 배정됐을 때

**dry-run에서 반드시 확인**: `bimamba_pure_*` 커맨드에 `--use_chunk_pairs`가 **없고** `--policy.sscp_enabled=false`가 **있어야** 한다. 이거 하나 틀리면 8시간을 날린다.

실행 셀은 잡이 끝날 때까지(~8h/잡) 블로킹한다.

In [ ]:
X.run_trains(plan['train'], plan['gpus'], dry=True)


In [ ]:
if X.role_of(NODE) != 'train':
    print('노드 ' + NODE + ' 는 eval 로 배정됐다 -> 6)번 eval 셀을 쓸 것. 여기는 건너뛴다.')
elif not plan['train']:
    print('학습 큐가 비었다.')
else:
    X.run_trains(plan['train'], plan['gpus'])


## 8) 이어서 — 다음 배치

위 배치가 끝나면 인벤토리가 바뀐다. 이 셀로 역할·계획을 다시 뽑고 6) 또는 7)로 돌아간다.

In [ ]:
# 위 배치가 끝나면 인벤토리가 바뀐다. 이 셀로 역할/계획을 다시 뽑고 6) 또는 7)로 돌아간다.
_ov = dict(X.ROLE_OVERRIDE)      # reload 하면 초기화되므로 수동 오버라이드를 보존한다
X = importlib.reload(X)
cf, v23 = X.setup(verbose=False)
X.ROLE_OVERRIDE = _ov
roles = X.suggest()
plan = X.plan(NODE, GPUS)


## 아침에 볼 것 — 토너먼트 판정

`X.report()` 가 TE 표 + 레짐 맵을 전부 찍는다. 판단 기준:

| 결과 | 다음 |
|---|---|
| `te_act_k20` **<** 71.4 | K=20에서 절대 우위 후보 확보 → K=10/15로 마진 확인 |
| `te_act_k10/15` vs `te_bimamba_k10/15`(순수) | **토너먼트 최종 판정.** 순수판이 이기면 W2 확보 |
| rm K=50 s10: `carry` vs `act` | 69.5/67.6 표 모순 해소 — **TE 없이** 이기면 최상 (동일 비용 승리) |
| act+TE가 짧은 K 전부 승 | W2 없음 → 긴 chunk 크로스오버(+효율) + real-world 스토리로 전환 |

**주의**: seed 1개 · 500ep 기준 binomial SE ≈ ±2.2%p. **5%p 미만 차이는 주장하지 말 것.** 이기는 지점이 나오면 그 셀만 seed/rep을 늘려 굳힌다.

In [ ]:
X.report()
